# Topological MQA (TMQA) 1: Simulation
Welcome to the first Jupyter Notebook for topological analysis of Mirror Quantum Awesomeness. The abbreviation will be Topological MQA, or TMQA. This notebook is about running different size of square lattice on the TMQA algorithm.

We tested the benchmarking process for 4x4, 6x6, 8x8 and 10x10 on classical computer (MacBook Pro M3 Max, 32GB RAM, 1TB SSD) and it spent few hours.

Thus, we would recommend running the simulation with these square lattice first. 

Key files are described as below:
- `qiskit_device_benchmarking/utilities/sampling_utils.py`: Find `TopoSampler` and its parent class `NewSampler`.
- `qiskit_device_benchmarking/bench_code/mrb/mirror_rb_experiment.py`: Find `if not self.experiment_options.full_sampling` in line 375. There is a workflow specially for `TopoSampler` and `NewSampler`.
- A class and its utility class for Topological MQA: `qiskit_device_benchmarking/bench_code/mrb/mirror_qa_topo.py`.

The simple logic is described as below:
- A researcher choose the error rate (`p2`), the depth of circuit that they want to investigate (`lengths`), the number of `samples` and `shots` that they want to use (the higher the better) and the size of a square lattice that they want to use. This supports not only `n x n` (where `n >= 2`) but for example, `10 x 12` like IBM's Nighthawk.
- Creation of `MirrorQATopo` object will store these information, access the sampling algorithm to randomly choose either: 
  - As maximum pairs as possible on that square lattice: `mode=full` on `TopoSampler` while connecting it to `NewSampler`, or
  - As maximum pairs as possible while *i*) abandoning one qubit per each 'boundary' (on the opposite edge of the coupling map) and *ii*) make the pair of 'imaginary qubit' that surrounds that coupling map to form a torus shape. This is `mode=random`.
  - If a researcher choose `mode=random` while they build `MirrorQATopo` object with `mode` parameter, the condition for `mode=full` and `mode=random` will equally appear for each layers.
- `TopoSampler` will generate not only the pattern above, but let two qubit gates (`CXGate`) will only appear in the even index of a circuit (with randomly-chosen possible pairs via MWPM), and randomly-chosen single qubit gates in the odd index of a circuit from each layer.
- The `MirrorQATopo` object will execute `.run()` that processes through all the circuits that it built.
- The result will be:
  - Effective Polarizer graph and Mutual Information graph: Same process as other MQA benchmarking process.
  - `P(pairs)` and `P(topo)`: The metric where the *bot*, who deducts the correct pair and the correct `mode` based on MI data, which is described as a probability of how *bot* deducted it accurately or inaccurately, per each.
  - The graph for this choose its y-axis as `1-P(pairs)` or `1-P(topo)` to demonstrate the *critical point* where the wrong answer from the puzzle suddenly explodes after when the system is randomly entangled too much to recover.

## Setup

In [ ]:
# import the necessary packages first, and setup the seed

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import os, random, gc, pickle, pathlib
from collections import defaultdict
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target
from qiskit.quantum_info import Operator
from qiskit_device_benchmarking.bench_code.mrb import (
    MirrorQATopo,
    QuantumAwesomeness,
    TopoUtil,
)

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Shared parameters for each lattice
basis_gates = ["id", "h", "x", "y", "z", "rz", "cx"]
p2 = 1e-2 # *** 1% error rate ***
# For 0.1%, 5%, 10%, p2 can be 1e-3, 5e-2 and 1e-1, respectively.
p1 = p2 / 10
rz_angle = np.pi / 2
shots = 1000

# *** Below are the suggestion to view the critical point well. ***
# For 0.1% error rate,
# For 1% error rate, [26, 40, 54, 68, 82] worked.
# For 5% error rate, [16, 18, 20, 22, 24] worked.
# For 10% error rate, [8, 10, 12, 14, 16] worked.
lengths = [16, 18, 20, 22, 24]
num_samples = 1000
num_lengths = len(lengths) # 5, in this case.

# *** Square lattice for benchmarking. Extend this with freedom! ***
sizes = [4, 6, 8, 10] # [4] means it will create 4x4 lattice

custom_name_mapping = {
    "id": Operator(np.eye(2)),
    "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)),
    "x": Operator(np.array([[0, 1], [1, 0]])),
    "y": Operator(np.array([[0, -1j], [1j, 0]])),
    "z": Operator(np.array([[1, 0], [0, -1]])),
    "rz": Operator(
        [ 
         [np.cos(rz_angle / 2), -1j * np.sin(rz_angle / 2)], 
         [-1j * np.sin(rz_angle / 2), np.cos(rz_angle / 2)],
        ]
    ),
    "cx": Operator(np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])),
}

noise_model = NoiseModel()
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)
for gate in basis_gates:
    if gate == "cx":
        noise_model.add_all_qubit_quantum_error(error_2q, gate)
    else:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)

### Helper functions

Below cells are for helper functions to:
1. Save the ongoing experiment as `.pkl` to the disk, so if the kernel crashes or the laptop goes to Sleep Mode, a researcher doesn't need to execute the entire experiment from the beginning again.
2. Analysis & Terminal printing: Definition of a *bot*, calculation of the `P(pairs)` and `P(topo)`.

In [ ]:
# Part 1: Prepare .pkl
ckpt_dir = pathlib.Path("topo_ckpt")
ckpt_dir.mkdir(exist_ok=True)

In [ ]:
# Part 2: Experiment side
# To deduct the mode for TopoSampler
def get_topo_mode(topo_full, fake_qubits):
    """f2f if the fake qubits paired together (or stayed unpaired); 
    f2g otherwise."""
    fake_pairs = [p for p in topo_full if any(q in fake_qubits for q in p)]
    if not fake_pairs:
        return "f2f"
    for p in fake_pairs:
        if all(q in fake_qubits for q in p):
            return "f2f"
    return "f2g"

In [ ]:
# Make the backend for each size. This goes in the loop.
def build_backend(n):
    """Build (backend, num_qubits, legit_qubits) 
    for an n x n genuine lattice."""
    cmap = TopoUtil.makeCouple(n, n, 2, "ibm")
    num_qubits = len(list(cmap.physical_qubits))
    legit_qubits = n * n
    target = Target.from_configuration(
        num_qubits=num_qubits,
        basis_gates=basis_gates,
        coupling_map=cmap,
        custom_name_mapping=custom_name_mapping,
    )
    backend = AerSimulator(
        method="stabilizer",
        noise_model=noise_model if (p1 > 0 or p2 > 0) else None,
        target=target,
        max_parallel_threads=0,
        max_parallel_experiments=0,
        seed_simulator=SEED,
    )
    return backend, num_qubits, legit_qubits

In [ ]:
# Run the experiment
def run_topo(backend, legit_qubits):
    """Build the MirrorQATopo experiment, run it, 
    and block until results land."""
    exp = MirrorQATopo(
        range(legit_qubits),
        lengths=lengths,
        sampling_algorithm="topo",
        mode="random",
        backend=backend,
        two_qubit_gate_density=0.5,
        num_samples=num_samples,
        initial_entangling_angle=np.pi / 2,
        seed=SEED,
    )
    exp.set_run_options(shots=shots)
    rb_data = exp.run()
    
    rb_data.block_for_results()
    return exp, rb_data

In [ ]:
# Calculates the MI from the result
def extract_mi(exp, rb_data, legit_qubits):
    legit_cmap = exp.backend.coupling_map.reduce(range(legit_qubits))
    qa = QuantumAwesomeness(legit_cmap)
    return qa.mutual_info(rb_data.data())

This is a bot.

In [ ]:
def evaluate_bot(mi, exp, legit_qubits, num_qubits, verbose=False):
    half = legit_qubits // 2
    fake_qubits = set(range(legit_qubits, num_qubits))

    bot_mode_ok = defaultdict(int)
    bot_pairs_ok = defaultdict(int)
    totals = defaultdict(int)

    for i, info in enumerate(mi):
        length = lengths[i % num_lengths]
        # sample = i // num_lengths

        topo_full = exp._topo_outcomes[i]
        topo_mode = get_topo_mode(topo_full, fake_qubits)

        G = nx.Graph()
        for (u, v), w in info.items():
            G.add_edge(u, v, weight=w)
        raw_guess = nx.max_weight_matching(G, maxcardinality=False, weight="weight")
        bot_pairs = sorted(tuple(sorted(p)) for p in raw_guess)
        bot_mode = "f2f" if len(bot_pairs) == half else "f2g"

        pairs_truth = sorted(tuple(sorted(p)) for p in exp._pairs[i])

        bot_mode_ok[length] += int(bot_mode == topo_mode)
        bot_pairs_ok[length] += int(set(bot_pairs) == set(pairs_truth))
        totals[length] += 1

    L = sorted(lengths)
    return {
        "lengths": L,
        "p_mode_topo": [bot_mode_ok[l] / totals[l] for l in L],
        "p_bot_pairs": [bot_pairs_ok[l] / totals[l] for l in L],
    }


## Build and run the `exp`

In [ ]:
# Prepare to run the experiment with the checkpoint management with .pkl
import hashlib, json

def _ckpt_path(n):
    key = dict(
        n=n,
        p1=p1,
        p2=p2,
        shots=shots,
        num_samples=num_samples,
        lengths=list(lengths),
        seed=SEED,
    )
    h = hashlib.md5(json.dumps(key, sort_keys=True).encode()).hexdigest()[:8]
    return ckpt_dir / f"lattice_{n}_p2-{p2:g}_{h}.pkl"

results = {}  # results[n] = {'lengths': [...], 'p_bot_pairs': [...], 'params': {...}}

In [ ]:
# For running multiple different sizes of lattice altogether

for n in sizes:
    ckpt = _ckpt_path(n)
    if ckpt.exists():
        results[n] = pickle.loads(ckpt.read_bytes())
        prm = results[n].get("params", {})
        # Guard against a hash collision / hand-edited file: verify provenance.
        if prm.get("p2") != p2:
            raise RuntimeError(
                f"Checkpoint {ckpt.name} has p2={prm.get('p2')} but current p2={p2}. "
                f"Delete it and rerun."
            )
        print(
            f"[{n}x{n}] loaded cache {ckpt.name}  "
            f"P(bot=pairs)={[f'{p:.3f}' for p in results[n]['p_bot_pairs']]}"
        )
        continue

    print(f"\n===== {n}x{n} Lattice (p2={p2:g}) =====")
    backend, num_qubits, legit_qubits = build_backend(n)

    exp, rb_data = run_topo(backend, legit_qubits)
    print(f"  ran {len(exp._pairs)} circuits  job_ids={rb_data.job_ids}")

    mi = extract_mi(exp, rb_data, legit_qubits)
    print(f"  extracted MI for {len(mi)} circuits") # An indicator

    results[n] = evaluate_bot(mi, exp, legit_qubits, num_qubits, verbose=False)
    
    results[n]["params"] = dict(
        n=n,
        p1=p1,
        p2=p2,
        shots=shots,
        num_samples=num_samples,
        lengths=list(lengths),
        seed=SEED,
    )
    # Sneak-peak of the result before draw the plot
    # We used this to estimate the critical point before the experiment ends, to control 'lengths' well.
    print(f"  P(bot=pairs) = {[f'{p:.3f}' for p in results[n]['p_bot_pairs']]}")

    ckpt.write_bytes(pickle.dumps(results[n]))
    print(f"  saved {ckpt.name}") # Another indicator for peace of mind...

    del exp, rb_data, mi, backend
    gc.collect()

# --- Final data-point summary (always shows, fresh or cached) -----------------
print(f"\n=== Data points  (p2={p2:g} = {p2 * 100:g}%) ===")
hdr_lens = results[sizes[0]]["lengths"]
print(f"{'lattice':>8} | " + " | ".join(f"L={L:>3}" for L in hdr_lens))
print("-" * (10 + 8 * len(hdr_lens)))
for n in sizes:
    row = results[n]["p_bot_pairs"]
    print(f"{f'{n}x{n}':>8} | " + " | ".join(f"{p:5.3f}" for p in row))

## Draw the final comparison plot

This one is for `P(pairs)`.

In [ ]:
# Redraw the plot based on the reference
markers = ["o"]
plt.figure(figsize=(7, 4.5))
for i, n in enumerate(sizes):
    y = [1 - p for p in results[n]["p_bot_pairs"]] 
    plt.plot(
        results[n]["lengths"],
        # results[n]["p_bot_pairs"],
        y,
        marker=markers[i % len(markers)],
        linestyle="-",
        label=f"L={n}",
    )

plt.xlabel("Circuit Depth (length)")
plt.ylabel(r"1-$P(\mathrm{pairs})$")

plt.grid(True, alpha=0.3)
plt.legend(loc="center left", frameon=True)
plt.tight_layout()
plt.show()

And this is for `P(topo)`.

In [ ]:
markers = ["o"]
plt.figure(figsize=(7, 4.5))
for i, n in enumerate(sizes):
    y = [1 - p for p in results[n]["p_mode_topo"]]
    plt.plot(
        results[n]["lengths"],
        # results[n]["p_bot_pairs"],
        y,
        marker=markers[i % len(markers)],
        linestyle="-",
        label=f"L={n}",
    )

plt.xlabel("Circuit Depth (length)")
plt.ylabel(r"1-$P(\mathrm{topo})$")

plt.grid(True, alpha=0.3)
plt.legend(loc="center left", frameon=True)
plt.tight_layout()
plt.show()